# In a nutshell

Compare how human HTML annotators stack against an LLM of your choice using different prompts and settings.

# Setup

In [13]:
import json
import os
import sys
from pathlib import Path

# Adjust working dir/Python path
# exactly once per session
# using IPython user namespace for run-once behaviour.
ipy = get_ipython()
if not ipy.user_ns.get("syspath_set", False):
    os.chdir("..")
    sys.path.append(".")
    ipy.user_ns["syspath_set"] = True

In [14]:
Path.cwd()

WindowsPath('d:/Projects/uoe-html-annotation/d2g-evaluation')

In [15]:
from dotenv import load_dotenv

# Load API key from .env file if present.
load_dotenv()

# Not recommended: set API key manually
# %env VAR_NAME=value

True

# Load the data

In [16]:
dataset_path = "data/eng_Latn_sample_for_llm_eval.parquet"

In [17]:
from datasets import Dataset

ds = Dataset.from_parquet(dataset_path).sort(column_names="task_id")
print(f"Dataset size: {len(ds)} docs")
print(f"Columns:{ds.column_names}")
print(ds[:1])

Dataset size: 25 docs
Columns:['task_id', 'file_name', 'html', 'language', 'annotations', 'annotation_count', 'filename_warc', 'url', 'timestamp', 'collection']
{'task_id': [181401785], 'file_name': ['89a71c87-1354_cc_m_uk.html'], 'html': ['\ufeff<!DOCTYPE html>\r\n\r\n\r\n\r\n<meta name="google-site-verification" content="6VN-GS2-Hd9k49JKgO_I-mDWpY6IB8Sx7CWkzLl_z9A" />\r\n<!-- <html> -->\r\n<html xmlns:fb="http://ogp.me/ns/fb#">\r\n<head>\r\n<link rel="stylesheet" href="/v/newspaper.css">\r\n<link rel="shortcut icon" href="http://www.leamingtonobserver.co.uk/favicon.ico">\r\n<meta http-equiv="content-type" content="text/html; charset=utf-8"/>\r\n<title>Top broadcasting award for Leamington student | Leamington Observer</title>\r\n<meta name="description" content="HIGHLIGHTING the issue of skin lightening in the Asian community has seen a budding TV presenter from Leamington win a top national award">\r\n<meta name="google-site-verification" content="6VN-GS2-Hd9k49JKgO_I-mDWpY6IB8Sx7CW

# Experiment configuration

In [18]:
# fmt: off
config = {
    "model": "deepseek/deepseek-r1-distill-qwen-32b",
    "connector": "openrouter",
    "html_denoiser": "light",
    "_html_denoiser_help": "Supported values: `light` | None. `light` strips inline styles, js, irrelevant tags to cut token costs and improve LLM annotation quality.",
    "concurrent_requests": 5,
    "prompt": {
      "_variant": "p2_nothink",
      "system": "Act as a veteran dataset annotator working for OpenAI. Your task is to annotate all main content text in given html. Output the identified main content text as JSONL object with a single key 'annotations'. The value of 'annotations' is a list in which each object is a single main content text span with a single field 'text'. The value of 'text' is the exact literal value of the identified main content text span.\n Hints:\n        - Main content text must be distinct from repetitive boilerplate and provide unique or meaningful information relevant to the page.\n        - Main content text includes headings, dates, locations, tables, comments, lists of items, item properties and specifications, item prices, item reviews, image captions, forum threads.\n        - Main content text generally excludes common boilerplate elements that appear on every page of a website such as navigation menus, sorting dropdowns, headers, and footers.\n        - Always try to annotate the longest continuous span.\n        - Annotate only the spans that you are sure about. If you are not sure about an annotation, skip it.\n        - If there is nothing to annotate in the html, 'annotations' will be an empty list.\n        - Respond with a plain Python-compatible string instantly pluggable into json.loads. Do not format or pretty-print the output. \n        - All fields are compulsory. Output nothing but the JSONL object.  Output valid JSON. Escape all quotes inside strings with backslash. Do not wrap response in code fences. Do not summarize. Do not omit anything. If output is long, that's fine. /no_think",
      "user": "Given the html: \n ```{html}```\n, annotate the main content text."
    },
    "reasoning": {
      "enabled": True
    },
    "provider": {
      "only": [
        "deepinfra/fp8"
      ]
    },
    "_provider_help": "Different providers may be using different quantizations and hardware, leading to unwanted quality fluctuations across runs. Use a fixed one for full run.",
    "name": "deepseek-r1-distill-qwen-32b-fp8-p2_no_think-light-denoiser-dev-scratch",
    "_name_help": "It's the experiment name used to determine the output folder. Required when passing config as a JSON string in Jupyter. When calling `llm_annotator.py` with a file system path, the output folder will be named after the config file."
}
# fmt: on

# Call LLM annotator


In [19]:
import subprocess

# fmt: off
args = [
    "--input",
    dataset_path,
    "--out_dir",
    "llm/.scratch",
    "--max_docs",  # cap the number of documents if you'd like.
    "5",
    "--n_runs",  # run multiple times to get averaged metrics.
    "2",
    "--config",
    json.dumps(config),
]
# fmt: on

process = subprocess.Popen(
    ["python", "-u", "-m", "llm.llm_annotator", *args],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end="")

process.wait()

17:55:31 [INFO] Loaded config {'model': 'deepseek/deepseek-r1-distill-qwen-32b', 'connector': 'openrouter', 'html_denoiser': 'light', '_html_denoiser_help': 'Supported values: `light` | None. `light` strips inline styles, js, irrelevant tags to cut token costs and improve LLM annotation quality.', 'concurrent_requests': 5, 'prompt': {'_variant': 'p2_nothink', 'system': "Act as a veteran dataset annotator working for OpenAI. Your task is to annotate all main content text in given html. Output the identified main content text as JSONL object with a single key 'annotations'. The value of 'annotations' is a list in which each object is a single main content text span with a single field 'text'. The value of 'text' is the exact literal value of the identified main content text span.\n Hints:\n        - Main content text must be distinct from repetitive boilerplate and provide unique or meaningful information relevant to the page.\n        - Main content text includes headings, dates, locati

0

# Evaluation

In [20]:
import polars as pl

from llm.storage import AnnotationRunStorage

storage = AnnotationRunStorage(base_path="llm/.scratch", config=json.dumps(config))

### Validate annotation experiment runs

- How many docs are annotated in each run so far?
- Ensure no contamination: did both config and html dataset remain unchanged across runs?

In [21]:
from llm.evaluators import RunStateReport

r = RunStateReport()
print(f"Experiment config: {config['name']}")
run_state = r.generate(storage)

Experiment config: deepseek-r1-distill-qwen-32b-fp8-p2_no_think-light-denoiser-dev-scratch
shape: (2, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ run_id         ┆ docs_processed ┆ config         ┆ input_source  ┆ config_hash   ┆ input_hash    │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│ str            ┆ i64            ┆ str            ┆ str           ┆ str           ┆ str           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ 20251215_21304 ┆ 5              ┆ deepseek-r1-di ┆ data/eng_Latn ┆ 0b5ca1a11e5dc ┆ 00a742fb22c66 │
│ 7              ┆                ┆ still-qwen-32b ┆ _sample_for_l ┆ 0cc           ┆ 5ef           │
│                ┆                ┆ -fp8-p2_no_thi ┆ lm_eval.parqu ┆               ┆               │
│                ┆                ┆ nk-light-denoi ┆ et            ┆               ┆   

#### Human vs LLM annotation metrics

- For each document, the annotations from each evaluator are concatenated into a single string.
- Pairwise LLM-vs-human precision/recall/f1 are computed via token matching;
- We average these metrics over all humans who annotated the document.

In [22]:
from llm.evaluators import HumanVsLlmExperimentMetricsReport

r = HumanVsLlmExperimentMetricsReport()

print(f"Experiment config: {config['name']}")
metrics = r.generate(storage=storage)

Experiment config: deepseek-r1-distill-qwen-32b-fp8-p2_no_think-light-denoiser-dev-scratch


Evaluating with metric 'lcs_token_matching':   0%|          | 0/5 [00:00<?, ? examples/s]

Evaluating with metric 'lcs_token_matching':   0%|          | 0/5 [00:00<?, ? examples/s]

shape: (3, 8)
┌───────────────┬──────┬───────────┬────────┬───────┬───────────────┬───────────────┬──────────────┐
│ run_id        ┆ docs ┆ precision ┆ recall ┆ f1    ┆ mean_doclengt ┆ mean_tokens_c ┆ compression_ │
│ ---           ┆ ---  ┆ ---       ┆ ---    ┆ ---   ┆ h_chars       ┆ onsumed       ┆ ratio        │
│ str           ┆ f64  ┆ f64       ┆ f64    ┆ f64   ┆ ---           ┆ ---           ┆ ---          │
│               ┆      ┆           ┆        ┆       ┆ f64           ┆ f64           ┆ f64          │
╞═══════════════╪══════╪═══════════╪════════╪═══════╪═══════════════╪═══════════════╪══════════════╡
│ 20251215_2130 ┆ 5.0  ┆ 0.84      ┆ 0.868  ┆ 0.826 ┆ 59684.0       ┆ 8071.0        ┆ 7.39         │
│ 47            ┆      ┆           ┆        ┆       ┆               ┆               ┆              │
│ 20251216_1103 ┆ 5.0  ┆ 0.71      ┆ 0.85   ┆ 0.7   ┆ 59684.0       ┆ 8071.0        ┆ 7.39         │
│ 32            ┆      ┆           ┆        ┆       ┆               ┆        

## Explore low-performers

In [23]:
run_id = storage.get_last_run_id(config=storage.run_config)
r = HumanVsLlmExperimentMetricsReport()
eval_stats, _, _ = r.evaluate_single_run(run_id, storage)

print(f"Experiment config: {config['name']}")
print(f"run_id: {run_id}")
low_performers = eval_stats.filter(pl.col("f1") < 0.7).sort("f1")  # noqa

with pl.Config(fmt_str_lengths=10**5, fmt_table_cell_list_len=10**5, tbl_cols=-1, tbl_rows=-1):
    print(low_performers["task_id", "precision", "recall", "f1", "llm_annotations", "human_annotations"])

Evaluating with metric 'lcs_token_matching':   0%|          | 0/5 [00:00<?, ? examples/s]

Experiment config: deepseek-r1-distill-qwen-32b-fp8-p2_no_think-light-denoiser-dev-scratch
run_id: 20251216_110332
shape: (2, 6)
┌───────────┬───────────┬────────┬───────┬────────────────────────────┬────────────────────────────┐
│ task_id   ┆ precision ┆ recall ┆ f1    ┆ llm_annotations            ┆ human_annotations          │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---                        ┆ ---                        │
│ i64       ┆ f64       ┆ f64    ┆ f64   ┆ list[str]                  ┆ list[str]                  │
╞═══════════╪═══════════╪════════╪═══════╪════════════════════════════╪════════════════════════════╡
│ 181405856 ┆ 0.126     ┆ 0.959  ┆ 0.222 ┆ ["Design Your Own!         ┆ ["Design Your Own!         │
│           ┆           ┆        ┆       ┆ Megaphone (1 dz)", "Posted ┆ Megaphone (1 dz)\nPosted   │
│           ┆           ┆        ┆       ┆ by cheaptoysforsale on     ┆ by cheaptoysforsale on     │
│           ┆           ┆        ┆       ┆ June 5, 2011 · Leave